In [6]:
import pandas as pd
import math

data = pd.read_csv("weather.csv")
target = "Play"

print("Dataset:")
print(data)

# CN2

def cn2(data):
    rules = []
    features = [c for c in data.columns if c != target]

    for feature in features:
        for value in data[feature].unique():

            subset = data[data[feature] == value]

            # Majority class
            result = subset[target].mode()[0]

            # Rule
            rules.append((feature, value, result))

    return rules


print("\nCN2 Classification Rules:")
cn2_rules = cn2(data)

for feature, value, result in cn2_rules:
    print(f"IF {feature} = {value} THEN {target} = {result}")

#  FOIL Algorithm

def foil(data, target_class):

    features = [c for c in data.columns if c != target]

    positive = data[data[target] == target_class]
    negative = data[data[target] != target_class]

    rules = []

    while len(positive) > 0:

        pos = positive.copy()
        neg = negative.copy()
        rule = []

        # Keep adding conditions until no negative example remains
        while len(neg) > 0:

            best = None
            best_gain = -1

            for feature in features:

                if feature in [x[0] for x in rule]:
                    continue

                for value in data[feature].unique():

                    new_pos = pos[pos[feature] == value]
                    new_neg = neg[neg[feature] == value]

                    p0, n0 = len(pos), len(neg)
                    p1, n1 = len(new_pos), len(new_neg)

                    if p1 == 0 or p1 + n1 == 0:
                        continue

                    # FOIL Information Gain
                    old = p0 / (p0 + n0)
                    new = p1 / (p1 + n1)

                    gain = p1 * math.log2(new / old)

                    if gain > best_gain:
                        best_gain = gain
                        best = (feature, value)

            if best is None:
                break

            # Add best condition to rule
            feature, value = best
            rule.append(best)

            # Keep only examples satisfying the condition
            pos = pos[pos[feature] == value]
            neg = neg[neg[feature] == value]

        if not rule:
            break

        rules.append((rule, target_class))

        # Remove positive examples already covered
        mask = pd.Series(True, index=positive.index)

        for feature, value in rule:
            mask &= positive[feature] == value

        positive = positive[~mask]

    return rules

#Learn rules for all classes

all_rules = []

for cls in data[target].unique():
    all_rules += foil(data, cls)

#Display rules

print("\nFOIL First-Order Rules:")

for rule, cls in all_rules:

    conditions = " AND ".join(
        f"{feature}({value})"
        for feature, value in rule
    )

    print(f"IF {conditions} THEN {target}({cls})")

# Prediction

def predict(row, rules):

    for rule, cls in rules:

        if all(row[f] == v for f, v in rule):
            return cls

    return "Unknown"


print("\nPredictions:")

for i, row in data.iterrows():

    result = predict(row, all_rules)

    print(
        f"Example {i+1}: "
        f"Actual = {row[target]}, "
        f"Predicted = {result}"
    )

Dataset:
     Outlook Temperature Humidity  Windy Play
0      Sunny         Hot     High  False   No
1      Sunny         Hot     High   True   No
2   Overcast         Hot     High  False  Yes
3       Rain        Mild     High  False  Yes
4       Rain        Cool   Normal  False  Yes
5       Rain        Cool   Normal   True   No
6   Overcast        Cool   Normal   True  Yes
7      Sunny        Mild     High  False   No
8      Sunny        Cool   Normal  False  Yes
9       Rain        Mild   Normal  False  Yes
10     Sunny        Mild   Normal   True  Yes
11  Overcast        Mild     High   True  Yes
12  Overcast         Hot   Normal  False  Yes
13      Rain        Mild     High   True   No

CN2 Classification Rules:
IF Outlook = Sunny THEN Play = No
IF Outlook = Overcast THEN Play = Yes
IF Outlook = Rain THEN Play = Yes
IF Temperature = Hot THEN Play = No
IF Temperature = Mild THEN Play = Yes
IF Temperature = Cool THEN Play = Yes
IF Humidity = High THEN Play = No
IF Humidity = Normal T